# Master End-to-End Analysis Notebook: EV Spare-Parts Import Prioritization

**Project:** EV Spare-Parts Import Prioritization for Addis Ababa, Ethiopia  
**Target Vehicles:** BYD Seagull (e-Platform 3.0), Toyota bZ4X (e-TNGA), Volkswagen ID.4 (MEB)  
**Pilot Capital Tranche:** USD 5,000–10,000  
**Slide 10 Deliverables Compliance Notebook:**
1. Raw Collected Data Ingestion
2. Data Cleaning & Part Normalization
3. Analysis & Modeling Pipeline
4. Source Log & Provenance Verification
5. Scoring Formula & Multi-Criteria Decision Analysis (MCDA)
6. Five Recommended Parts for Pilot Import
7. Limitations & Uncertainty Audit

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

# Set project paths
project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(project_root))

from src.config import RAW_DATA_DIR, PROCESSED_DATA_DIR, RANKINGS_DIR, BASELINE_SCORING_WEIGHTS
print("Project Root:", project_root)
print("Approved Baseline Scoring Weights:", BASELINE_SCORING_WEIGHTS)

## 1. Decision 1: Candidate Vehicle Evaluation & Selection

> **Methodological Boundary (Slide 4 & 6 Notes):**  
> Marketplace listings (Adey Market, Gezat Electric) reflect commercial visibility and dealer stocking intent only. They are never conflated with official registration census data.

In [ ]:
visibility_df = pd.read_csv(RAW_DATA_DIR / 'vehicle_visibility_raw.csv')
print(f"Raw visibility records: {len(visibility_df)}")
print("Selected Vehicle Triad:")
selected_vehicles = visibility_df[visibility_df['vehicle_id'].isin(['VEH-BYD-SGL', 'VEH-TOY-BZ4X', 'VEH-VW-ID4'])]
selected_vehicles[['vehicle_id', 'make', 'model', 'observed_platform', 'source_portal']].drop_duplicates()

## 2. Decisions 2 & 3: 10 Part Categories & Data Cleaning

- Normalizing part numbers using FAPI-inspired alphanumeric stripping
- Converting international supplier currencies (CNY, AED) to standardized USD
- Logging all missing fields into `null_value_audit_log.csv` (enforcing zero fabrication)

In [ ]:
cleaned_df = pd.read_csv(PROCESSED_DATA_DIR / 'cleaned_dataset.csv')
null_log = pd.read_csv(PROCESSED_DATA_DIR / 'null_value_audit_log.csv')
print(f"Cleaned dataset contains {len(cleaned_df)} candidate SKUs across 10 categories.")
print(f"Null Audit Log tracks {len(null_log)} unverified/missing parameters.")
cleaned_df[['record_id', 'vehicle_model', 'part_category', 'clean_oem_part_number', 'price_usd', 'availability_status']].head(10)

## 3. Decision 4: Scoring Formula & Reconciled MCDA Execution

Composite Priority Score formula:
$$\text{Score} = 0.25 \cdot F_1 + 0.25 \cdot F_2 + 0.20 \cdot F_3 + 0.20 \cdot F_4 + 0.10 \cdot F_5$$

- $F_1$ (25%): Demand Evidence (Wear schedules + NHTSA defect signals)
- $F_2$ (25%): International Procurement Delay (Stockout severity)
- $F_3$ (20%): Vehicle Compatibility (Modular platform sharing)
- $F_4$ (20%): Sourcing Practicality (Acquisition price & shipping density)
- $F_5$ (10%): Evidence Confidence (Dual-source verification)

In [ ]:
scoring_df = pd.read_csv(PROCESSED_DATA_DIR / 'ev_parts_master_dataset.csv')
full_rankings = pd.read_csv(RANKINGS_DIR / 'full_ranking.csv')
print(f"Ranked dataset loaded: {len(full_rankings)} rows")
full_rankings[['primary_rank', 'sku_id', 'vehicle_make', 'vehicle_model', 'part_category', 'primary_composite_score']].head(10)

## 4. Decision 5: Five Recommended Parts for Pilot Import

In [ ]:
top_5 = pd.read_csv(RANKINGS_DIR / 'final_top_five.csv')
print("=== FINAL TOP 5 RECOMMENDED SKUs (PROVISIONAL) ===")
for idx, row in top_5.iterrows():
    print(f"[{row['baseline_rank_str']}] {row['candidate_id']} | {row['vehicle_make']} {row['vehicle_model']} {row['part_category']} | Baseline Score: {row['baseline_cps']:.2f}")
    print(f"      OEM Part #: {row['clean_oem_part_number']} | FOB Price: ${row['catalog_price_usd']} | Supplier: {row['supplier_name']}")

## 5. Sensitivity Analysis: 4 Strategic Scenarios

- Baseline (25/25/20/20/10)
- Demand-Focused (35/20/15/20/10)
- Evidence-Confidence-Focused (20/20/15/15/30)
- Sourcing-Practicality-Focused (20/20/15/35/10)

In [ ]:
sensitivity_df = pd.read_csv(RANKINGS_DIR / 'sensitivity_analysis.csv')
print(f"Sensitivity analysis dataset loaded: {len(sensitivity_df)} records")
sensitivity_df[['sku_id', 'vehicle_model', 'part_category', 'baseline_score', 'baseline_rank', 'demand_focused_score', 'demand_focused_rank', 'evidence_focused_score', 'sourcing_focused_score']].head(8)

## 6. Disclosed Limitations & Unavailable Data Audit

1. **No Ethiopian Vehicle Registration Census:** Marketplace listings show dealer intent, not on-road fleet density.
2. **Wholesale Landed Cost Gaps:** Catalog prices exclude ocean container shipping, Djibouti port clearance, and Ethiopian customs duties.
3. **Single-Supplier Price Points:** 29 priced SKUs rely on single-catalog observations.
4. **Geographic Asymmetry:** NHTSA safety recall data applies to US models, not Chinese domestic EVs.
5. **Year & Trim Variances:** Production batch differences between Chinese and European market variants.